# 02 — SQL analysis

Questions below use the SQLite file from the cleaning notebook (`data/tmdb.db`). Where an average uses budget or revenue, I ignore 0. Those zeros mean "unknown" on TMDB, and averaging them in would pretend a lot of films cost nothing.


In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 20)

conn = sqlite3.connect("data/tmdb.db")


## 1. Action movies released after 2015

List title and release date. Join movies to genres through the bridge table.


In [2]:
pd.read_sql(
    """
    SELECT m.title, m.release_date
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON g.genre_id = mg.genre_id
    WHERE g.genre_name = 'Action'
      AND m.release_date > '2015-12-31'
    ORDER BY m.release_date
    """,
    conn,
)


,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
...,...,...
263,F1,2025-06-25
264,Superman,2025-07-09
265,The Fantastic 4: First Steps,2025-07-23
266,Predator: Badlands,2025-11-05


268 action movies have a release date after 2015. The list opens with January and February 2016 (The 5th Wave, Kung Fu Panda 3, Deadpool). Action is not a small tag in the recent part of this file.


## 2. Top 10 highest-grossing movies, with genres

`GROUP_CONCAT` keeps this to one row per movie. Without it the join would repeat Avatar once per genre and the LIMIT would be wrong.


In [3]:
pd.read_sql(
    """
    SELECT m.title,
           m.revenue,
           GROUP_CONCAT(g.genre_name, ', ') AS genres
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON g.genre_id = mg.genre_id
    GROUP BY m.movie_id
    ORDER BY m.revenue DESC
    LIMIT 10
    """,
    conn,
)


,title,revenue,genres
0,Avatar,2923706026,"Action, Science Fiction, Adventure"
1,Avengers: Endgame,2799439100,"Adventure, Science Fiction, Action"
2,Avatar: The Way of Water,2334484620,"Action, Adventure, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction"
5,Avengers: Infinity War,2052415039,"Adventure, Action, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Action, Adventure, Science Fiction"
7,Zootopia 2,1868208796,"Animation, Adventure, Comedy, Mystery, Family"
8,Inside Out 2,1698863816,"Animation, Adventure, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Science Fiction, Thriller"


Avatar is first at about $2.92 billion, then Avengers: Endgame and Avatar: The Way of Water. Eight of the ten are adventure / action / science fiction. Titanic is the one that isn't (drama, romance). If I were looking at commercial upside in this dataset, that cluster is the pattern, not a one-off title.


## 3. Average budget and average revenue by genre

Sorted by average revenue. A movie in three genres is counted in all three, which is what the question asks, but it does mean the genre totals are not a partition of the movies.


In [4]:
pd.read_sql(
    """
    SELECT g.genre_name,
           ROUND(AVG(m.budget)) AS avg_budget,
           ROUND(AVG(m.revenue)) AS avg_revenue,
           COUNT(*) AS n_movies
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON g.genre_id = mg.genre_id
    WHERE m.budget > 0
      AND m.revenue > 0
    GROUP BY g.genre_name
    ORDER BY avg_revenue DESC
    """,
    conn,
)


,genre_name,avg_budget,avg_revenue,n_movies
0,Adventure,111393307.0,401451192.0,655
1,Animation,88282289.0,382103463.0,231
2,Family,88408767.0,357870775.0,329
3,Science Fiction,97864404.0,335689706.0,446
4,Action,94651416.0,309125778.0,784
5,Fantasy,89183665.0,307912995.0,378
6,Comedy,54652525.0,223039972.0,721
7,Music,42817021.0,205843825.0,47
8,Romance,37226946.0,177226878.0,320
9,Thriller,50844282.0,170049287.0,635


Adventure has the highest average revenue, about $401 million, from 655 movies with both figures filled in. Animation and family are next. Horror's average budget is about $26 million, much lower than action or adventure, and its average revenue is lower too.

I would not trust the TV Movie row. It is one film. Documentary never shows up here because it doesn't have both a budget and a revenue above zero.


## 4. Top 10 actors by number of movies


In [5]:
pd.read_sql(
    """
    SELECT actor_name,
           COUNT(DISTINCT movie_id) AS movie_count
    FROM cast
    GROUP BY person_id, actor_name
    ORDER BY movie_count DESC
    LIMIT 10
    """,
    conn,
)


,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Scarlett Johansson,32
6,Willem Dafoe,32
7,Mark Wahlberg,32
8,Morgan Freeman,31
9,Tom Cruise,31


Samuel L. Jackson is in 39 movies in this extract, then Brad Pitt and Robert De Niro at 38. I grouped by `person_id` as well as the name so two people with the same name wouldn't get merged. This list is mostly very famous leads. The file is skewed toward well-known films, so "most credits" here is not the same thing as "working actor who does the most films."


## 5. Directors with more than 3 movies

`job = 'Director'` only. Screenplay and Writer credits are in the same table and would inflate the counts.


In [6]:
directors = pd.read_sql(
    """
    SELECT person_name,
           COUNT(DISTINCT movie_id) AS movie_count
    FROM crew
    WHERE job = 'Director'
    GROUP BY person_id, person_name
    HAVING movie_count > 3
    ORDER BY movie_count DESC
    """,
    conn,
)
print(len(directors), "directors")
directors


217 directors


,person_name,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Robert Eggers,4
213,Ryan Coogler,4
214,Andy Muschietti,4
215,Wes Ball,4


217 directors clear the "more than 3" line. That bar is low once you have 2,500 movies, so the useful part is the top of the table: Steven Spielberg 27, Tim Burton and Ridley Scott 19, Martin Scorsese 16. Christopher Nolan is further down at 11, which is still a lot, just not the maximum.


## 6. Top 10 keywords


In [7]:
pd.read_sql(
    """
    SELECT keyword_name,
           COUNT(*) AS times_used
    FROM movie_keywords
    GROUP BY keyword_id, keyword_name
    ORDER BY times_used DESC
    LIMIT 10
    """,
    conn,
)


,keyword_name,times_used
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


The most common keyword is "based on novel or book" (449), then "sequel" (416). After that come the credits-stinger tags, which are a TMDB habit more than a story theme. If I were using keywords for "movies like this," I would down-weight sequel and stinger tags. They mark a format, not a plot.


## 7. Budget above average, revenue below average

Both comparisons are subqueries. Averages skip zeros. The movie also has to have both figures filled in, otherwise a 0 revenue would look like a flop when it is just missing.


In [8]:
under = pd.read_sql(
    """
    SELECT title, budget, revenue
    FROM movies
    WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
      AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
      AND budget > 0
      AND revenue > 0
    ORDER BY budget DESC
    """,
    conn,
)
print(len(under), "movies")
print("avg budget used in the subquery is about",
      round(pd.read_sql("SELECT AVG(budget) AS a FROM movies WHERE budget > 0", conn)["a"][0]))
print("avg revenue used in the subquery is about",
      round(pd.read_sql("SELECT AVG(revenue) AS a FROM movies WHERE revenue > 0", conn)["a"][0]))
under


273 movies
avg budget used in the subquery is about 60816431
avg revenue used in the subquery is about 223278963


,title,budget,revenue
0,The Marvels,274800000,206136825
1,Red One,250000000,185700759
2,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,Green Lantern,200000000,219851172
4,Mulan,200000000,69965374
...,...,...,...
268,Rambo III,63000000,189015611
269,Æon Flux,62000000,53321673
270,The Equalizer 2,62000000,190400157
271,Shooter,61000000,95700000


273 movies sit above the average budget (about $61 million) and below the average revenue (about $223 million). A few of the expensive ones, The Gray Man, Red Notice, The Irishman, have revenues that are tiny next to a nine-figure budget. Some of that is streaming titles, where "revenue" in TMDB is not really a box-office number. I would not hand this list to a studio as a flop list without checking which films were theatrical.


## 8. Actors in a Christopher Nolan film

Nolan is in the crew table, so this is a join from cast to crew on `movie_id`, limited to rows where his job is Director.


In [9]:
nolan_cast = pd.read_sql(
    """
    SELECT DISTINCT c.actor_name
    FROM cast c
    JOIN crew cr ON c.movie_id = cr.movie_id
    WHERE cr.person_name = 'Christopher Nolan'
      AND cr.job = 'Director'
    ORDER BY c.actor_name
    """,
    conn,
)
print(len(nolan_cast), "actors")
nolan_cast


84 actors


,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


84 distinct actors appear in the 11 films Nolan directed in this dataset. The list is long because I kept everyone in the top billed cast, not just the lead. Names like Michael Caine show up, which fits, but so do a lot of smaller billing credits. For a casting check I would also filter `cast_order` to 0–2 if I only wanted the leads.


## 9. Genre with the highest average rating

Only genres with at least 20 movies. I returned the full ranking, not just the winner, so the gap is visible.


In [10]:
pd.read_sql(
    """
    SELECT g.genre_name,
           ROUND(AVG(m.vote_average), 2) AS avg_rating,
           COUNT(DISTINCT m.movie_id) AS n_movies
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON g.genre_id = mg.genre_id
    GROUP BY g.genre_name
    HAVING n_movies >= 20
    ORDER BY avg_rating DESC
    """,
    conn,
)


,genre_name,avg_rating,n_movies
0,War,7.36,85
1,History,7.34,112
2,Music,7.32,49
3,Drama,7.25,937
4,Animation,7.20,246
5,Western,7.13,33
6,Romance,7.03,349
7,Crime,7.01,387
8,Family,6.94,344
9,Mystery,6.88,216


War has the highest average rating, 7.36 across 85 movies, with history just behind at 7.34. Action, which dominated the revenue table, is near the bottom of this one. The spread is not huge (horror is about 6.6), but it is enough to say that the genres people rate highest here are not the genres that gross the most.


## 10. Top 10 movies by number of keywords


In [11]:
pd.read_sql(
    """
    SELECT m.title,
           COUNT(k.keyword_id) AS keyword_count
    FROM movies m
    JOIN movie_keywords k ON m.movie_id = k.movie_id
    GROUP BY m.movie_id
    ORDER BY keyword_count DESC
    LIMIT 10
    """,
    conn,
)


,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,Pacific Rim: Uprising,49


Smile 2 has 98 keywords, then Taken 3 with 70. I don't think that means those films are thematically richer. Keyword counts in this file look like how thoroughly someone tagged the title. I wouldn't use the raw count as a quality or complexity measure. It is fine as a "which rows are heavily tagged" check, which is all this question asks.
